In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL2FydW5nb3BhbGFuL3JlcG9zL21vZm9ncmFwaHkvZG9jcw=='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/importlib/_bootstrap.py": 1756909683.181072, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/importlib/_bootstrap_external.py": 1756909683.1840236, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/zipimport.py": 1756909682.1390634, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/codecs.py": 1756909681.7841098, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/encodings/aliases.py": 1756909682.4869266, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/encodings/__init__.py": 1756909682.467157, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/encodings/utf_8.py": 1756909682.8301048, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/abc.py": 1756909681.751164, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/io.py": 1756909681.8854132, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/stat.py": 1756909682.0489361, "/Users/arungopalan/miniforge

In [2]:
import mofography as mgr
from mofography.vdw_radii import (
    default_vdw_radii_angstrom,    # Bondi / CRC
    mendeleev_vdw_radii_angstrom,  # CRC / Mendeleev
    typing_vdw_radii_angstrom,     # Bondi/CRC + Mendeleev fill for NaN entries
)
from ase.io import read
import numpy as np

In [3]:
from ase import Atoms

# One atom of each element of interest
PROBE = Atoms(
    symbols=["H", "C", "N", "O", "Zn", "Cu", "Zr"],
    positions=[(i * 3.0, 0, 0) for i in range(7)],
    cell=[30, 5, 5],
    pbc=True,
)

bondi = default_vdw_radii_angstrom(PROBE)
mend = mendeleev_vdw_radii_angstrom(PROBE)

# Alvarez (2013) — picometers, vendored here for self-containment
ALVAREZ_PM = {
    "H": 120.0, "C": 177.0, "N": 166.0, "O": 150.0, "F": 146.0,
    "Cu": 238.0, "Zn": 239.0, "Zr": 252.0,
}
alvarez = np.array([ALVAREZ_PM.get(s, 170.0) for s in PROBE.get_chemical_symbols()]) / 100.0

In [4]:
print(f"{'Element':>8} {'Bondi/CRC (Å)':>14} {'CRC/Mendeleev (Å)':>19} {'Alvarez (Å)':>13} {'Δ(A-B)':>8} {'Δ(C-B)':>8}")
print("-" * 76)
for i, sym in enumerate(PROBE.get_chemical_symbols()):
    a, b, c = bondi[i], mend[i], alvarez[i]
    print(f"{sym:>8} {a:>14.2f} {b:>19.2f} {c:>13.2f} {b-a:>+8.2f} {c-a:>+8.2f}")

 Element  Bondi/CRC (Å)   CRC/Mendeleev (Å)   Alvarez (Å)   Δ(A-B)   Δ(C-B)
----------------------------------------------------------------------------
       H           1.20                1.10          1.20    -0.10    +0.00
       C           1.70                1.70          1.77    +0.00    +0.07
       N           1.55                1.55          1.66    +0.00    +0.11
       O           1.52                1.52          1.50    +0.00    -0.02
      Zn           1.73                2.01          2.39    +0.28    +0.66
      Cu           1.70                1.96          2.38    +0.26    +0.68
      Zr           2.12                2.23          2.52    +0.11    +0.40


In [5]:
atoms = read("IRMOF-1.cif")
print(f"Atoms: {len(atoms)}  ({atoms.get_chemical_formula()})")
print(f"Cell:  {atoms.cell.cellpar()[:3].round(2)} Å (cubic)")

spacing = 0.5  # Å — the canonical default

Atoms: 424  (C192H96O104Zn32)
Cell:  [25.83 25.83 25.83] Å (cubic)


/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/site-packages/ase/io/cif.py:411: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(


In [6]:
# Bondi/CRC: dgrid_from_atoms_rust uses this by default
dgrid_bondi = mgr.dgrid_from_atoms_rust(atoms, spacing=spacing)

# CRC / Mendeleev: pass the explicit radii array
radii_crc = mendeleev_vdw_radii_angstrom(atoms)
dgrid_crc = mgr.dgrid_from_atoms_rust(atoms, spacing=spacing, radii=radii_crc)

# Alvarez: same pattern, custom array
radii_alvarez = np.array([ALVAREZ_PM.get(s, 170.0) for s in atoms.get_chemical_symbols()]) / 100.0
dgrid_alvarez = mgr.dgrid_from_atoms_rust(atoms, spacing=spacing, radii=radii_alvarez)

print(f"bondi    dgrid: shape={dgrid_bondi.shape}, void max={dgrid_bondi.max():.2f} Å, "
      f"void voxels (d>1): {(dgrid_bondi > 1.0).sum():,}")
print(f"crc      dgrid: shape={dgrid_crc.shape}, void max={dgrid_crc.max():.2f} Å, "
      f"void voxels (d>1): {(dgrid_crc > 1.0).sum():,}")
print(f"alvarez  dgrid: shape={dgrid_alvarez.shape}, void max={dgrid_alvarez.max():.2f} Å, "
      f"void voxels (d>1): {(dgrid_alvarez > 1.0).sum():,}")

bondi    dgrid: shape=(52, 52, 52), void max=7.54 Å, void voxels (d>1): 83,608
crc      dgrid: shape=(52, 52, 52), void max=7.54 Å, void voxels (d>1): 84,136
alvarez  dgrid: shape=(52, 52, 52), void max=7.47 Å, void voxels (d>1): 81,440


[RAYON] current_num_threads = 14
[RAYON] current_num_threads = 14
[RAYON] current_num_threads = 14


In [7]:
def run_seg(dgrid, atoms, *, vdw_set, radii_for_audit=None):
    return mgr.do_periodic_segmentation_rust(
        dgrid, atoms,
        mask_thickness=0.05,
        h=0.2,
        min_distance=1.5,
        threshold_abs=1.5,
        spacing=spacing,
        compute_connections=True,
        watershed_backend="rust_det",
        disable_tqdm=True,
        vdw_radii_set=vdw_set,
        vdw_radii_angstrom=radii_for_audit,
    )

seg_bondi = run_seg(dgrid_bondi, atoms, vdw_set="bondi")
seg_crc = run_seg(dgrid_crc, atoms, vdw_set="crc", radii_for_audit=radii_crc)
seg_alvarez = run_seg(dgrid_alvarez, atoms, vdw_set="alvarez", radii_for_audit=radii_alvarez)

[RAYON] current_num_threads = 14
/Users/arungopalan/repos/mofography/mofography/helper.py:546: RuntimeWarning: overflow encountered in scalar subtract
  pos_minus[dim] = max(pos_minus[dim] - 1, 0)


[pocket_maxima_check] Region 1: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 3: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 4: pocket_radius from maxima_info (5.284 Å) underestimates true region max (5.629 Å) by 0.345 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 5: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 9: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


... (32 outputs left out)

[pocket_maxima_check] Region 22: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 23: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 24: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 25: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


In [8]:
print(f"{'Table':<10} {'maxima_pbc':>12} {'all_maxima':>12}")
print("-" * 36)
for name, seg in [("bondi", seg_bondi), ("crc", seg_crc), ("alvarez", seg_alvarez)]:
    print(f"{name:<10} {len(seg['maxima_pbc']):>12} {len(seg['all_maxima']):>12}")

Table        maxima_pbc   all_maxima
------------------------------------
bondi                 8          125
crc                   8          125
alvarez               8          125


In [9]:
print(f"{'Table':<10} {'min peak r (Å)':>16} {'mean peak r (Å)':>18} {'max peak r (Å)':>16}")
print("-" * 64)
for name, dgrid, seg in [
    ("bondi", dgrid_bondi, seg_bondi),
    ("crc", dgrid_crc, seg_crc),
    ("alvarez", dgrid_alvarez, seg_alvarez),
]:
    # maxima_pbc are (N, 3) integer voxel indices into the dgrid
    m = seg["maxima_pbc"]
    peaks = dgrid[m[:, 0], m[:, 1], m[:, 2]]
    print(f"{name:<10} {peaks.min():>16.3f} {peaks.mean():>18.3f} {peaks.max():>16.3f}")

Table        min peak r (Å)    mean peak r (Å)   max peak r (Å)
----------------------------------------------------------------
bondi                 5.284              6.357            7.538
crc                   5.384              6.407            7.538
alvarez               5.284              6.322            7.468


In [10]:
def compute_pld(seg):
    """Smallest window diameter in the unit cell."""
    if not seg["connections"]:
        return float("nan")
    unique = set(seg["unique_regions"])
    diams = []
    for c in seg["connections"]:
        # c[0] = (region_a, region_b); c[2] in {"internal", "periodic"}
        if np.any(np.isin(c[0], list(unique))):
            diams.append(2.0 * float(c[4][0]))
    return min(diams) if diams else float("nan")

print(f"{'Table':<10} {'PLD (Å)':>12} {'# unit-cell windows':>22}")
print("-" * 48)
for name, seg in [("bondi", seg_bondi), ("crc", seg_crc), ("alvarez", seg_alvarez)]:
    n_uc = sum(1 for c in seg["connections"] if np.any(np.isin(c[0], list(seg["unique_regions"]))))
    print(f"{name:<10} {compute_pld(seg):>12.3f} {n_uc:>22}")

Table           PLD (Å)    # unit-cell windows
------------------------------------------------
bondi             7.356                    108
crc               7.358                    108
alvarez           7.218                    108


In [11]:
def void_volume(dgrid, spacing):
    return float((dgrid > 1.0).sum()) * (spacing ** 3)

print(f"{'Table':<10} {'void volume (Å³)':>18} {'relative to bondi':>20}")
print("-" * 52)
v_b = void_volume(dgrid_bondi, spacing)
for name, dgrid in [("bondi", dgrid_bondi), ("crc", dgrid_crc), ("alvarez", dgrid_alvarez)]:
    v = void_volume(dgrid, spacing)
    print(f"{name:<10} {v:>18.1f} {v / v_b:>20.4f}")

Table        void volume (Å³)    relative to bondi
----------------------------------------------------
bondi                 10451.0               1.0000
crc                   10517.0               1.0063
alvarez               10180.0               0.9741


In [12]:
for name, seg in [("bondi", seg_bondi), ("crc", seg_crc), ("alvarez", seg_alvarez)]:
    ip = seg["input_params"]
    print(f"\n{name} audit:")
    print(f"  vdw_radii_set:         {ip.get('vdw_radii_set')}")
    print(f"  vdw_radii_by_symbol:   {ip.get('vdw_radii_by_symbol')}")


bondi audit:
  vdw_radii_set:         bondi_crc_fallback
  vdw_radii_by_symbol:   {'Zn': 1.73, 'O': 1.52, 'C': 1.7, 'H': 1.2}

crc audit:
  vdw_radii_set:         crc
  vdw_radii_by_symbol:   {'Zn': 2.01, 'O': 1.52, 'C': 1.7, 'H': 1.1}

alvarez audit:
  vdw_radii_set:         alvarez
  vdw_radii_by_symbol:   {'Zn': 2.39, 'O': 1.5, 'C': 1.77, 'H': 1.2}


In [13]:
# One line per table, one column per probe radius
# (Useful as a quick figure for a paper or report)
import pandas as pd
rows = []
for name, dgrid in [("bondi", dgrid_bondi), ("crc", dgrid_crc), ("alvarez", dgrid_alvarez)]:
    for probe in [1.0, 1.5, 2.0, 2.5]:
        n_vox = int((dgrid > probe).sum())
        rows.append({"table": name, "probe_a": probe, "n_voxels": n_vox,
                     "volume_a3": n_vox * spacing**3})
df = pd.DataFrame(rows).pivot(index="table", columns="probe_a", values="volume_a3")
print("Accessible volume (Å³) at each probe radius:")
print(df.round(1))

Accessible volume (Å³) at each probe radius:
probe_a      1.0     1.5     2.0     2.5
table                                   
alvarez  10180.0  8292.0  6485.0  4785.0
bondi    10451.0  8509.0  6638.0  4911.0
crc      10517.0  8638.0  6743.0  5004.0
